# Lab 4 — PyTorch Linear Regression

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 4</strong></h4>
<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Train a linear regression model in PyTorch using a regression dataset. Use the following parameters.
</p>
<ul>
    <li>Criterion: MSE Loss</li>
    <li>Fully Connected Layers x 2</li>
    <li>Batch Size: 8</li>
    <li>Optimizer: SGD</li>
    <li>Epoch: 1000</li>
</ul>
</div>

### My thought process

I need to build an end-to-end PyTorch training pipeline for a regression problem. Let me break the requirements down into concrete pieces before I start coding:

- **A regression dataset.** I don't have a specific dataset handed to me, so I'll generate a synthetic one with `sklearn.datasets.make_regression`. That gives me continuous features and a continuous target, which is exactly what I need for regression, and it keeps this notebook self-contained since I don't have to upload any external files.
- **"Fully Connected Layers x 2".** I'm reading this as: my model should stack **two `nn.Linear` layers**. If I only used one `nn.Linear` layer, that would literally just be plain linear regression with no room to show a proper 2-FC-layer model, so I'll put a ReLU in between the two layers — that keeps it a small feed-forward network while still satisfying "2 FC layers."
- **Batch Size 8** → I'll hold out part of the data for testing, and load the training data in mini-batches of 8 using `Dataset` + `DataLoader`.
- **Criterion: MSE Loss** → `nn.MSELoss()`, the standard loss for regression.
- **Optimizer: SGD** → `torch.optim.SGD`.
- **1000 epochs** → I'll loop over the full training set 1000 times, and track the loss along the way so I can plot the learning curve at the end and actually verify convergence instead of just assuming it happened.

**My pipeline:**
1. Standard imports + set a seed so my results are reproducible.
2. Generate & inspect the regression dataset; scale the features/target (this should help SGD converge more smoothly) and split into train/test.
3. Wrap the arrays in a PyTorch `Dataset`/`DataLoader` with `batch_size=8`.
4. Define the 2-FC-layer model, the `MSELoss` criterion, and the `SGD` optimizer.
5. Train for 1000 epochs, printing the loss periodically so I can watch it drop.
6. Plot the loss curve and check performance on the held-out test set.


In [ ]:
# ---- standard imports ----
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# setting a seed so I get the same results every time I rerun this notebook
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)

### 1. Creating my regression dataset

I'll generate a synthetic dataset with a handful of informative features plus some Gaussian noise, so the task is learnable but not trivial — if there's no noise at all, the model would converge to near-zero loss almost immediately and I wouldn't really see a learning curve.

In [ ]:
X, Y = make_regression(
    n_samples=1000,
    n_features=5,
    n_informative=5,
    noise=15.0,
    random_state=SEED,
)
Y = Y.reshape(-1, 1)   # I want my target as a column vector, shape (n_samples, 1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("First row of X:", X[0])
print("First target  :", Y[0])

### 2. Train/test split and feature scaling

I'm holding out 20% of the data so I have something to evaluate on later that the model hasn't seen. I'm also standardizing the features and the target — I fit the scaler on the training data only, then apply it to both splits, since fitting it on the test set too would be leaking information. This should keep the values in a range that trains smoothly with plain SGD (no fancy adaptive optimizer to compensate for badly-scaled inputs).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=SEED
)

x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)

X_train = x_scaler.transform(X_train)
X_test = x_scaler.transform(X_test)
y_train = y_scaler.transform(y_train)
y_test = y_scaler.transform(y_test)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

### 3. PyTorch `Dataset` and `DataLoader` (batch size = 8)

I need a `Dataset` so `DataLoader` knows how to grab individual samples and batch them for me.

In [ ]:
class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = RegressionDataset(X_train, y_train)
test_dataset = RegressionDataset(X_test, y_test)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# quick sanity check before I move on -- let me peek at one batch
xb, yb = next(iter(train_loader))
print("Batch X shape:", xb.shape)
print("Batch y shape:", yb.shape)

### 4. Model, criterion, optimizer

I'm defining two fully connected layers: an input layer that projects my 5 features into a hidden representation, and an output layer that maps that hidden representation down to a single regression output. I put a ReLU between them so the two `Linear` layers aren't mathematically equivalent to just one — otherwise stacking two linear layers with nothing in between would just collapse into one linear transformation.

In [ ]:
class LinearRegressionNet(nn.Module):
    def __init__(self, in_features, hidden_features=16):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)   # FC layer 1
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, 1)              # FC layer 2 (output)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

n_features = X_train.shape[1]
model = LinearRegressionNet(in_features=n_features, hidden_features=16)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

### 5. Training loop (1000 epochs)

This is the standard PyTorch training loop pattern: zero the gradients, forward pass, compute loss, backward pass, step the optimizer. I'm accumulating the per-batch loss so I can report an average loss per epoch.

In [ ]:
EPOCHS = 1000
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()          # reset gradients from the previous step
        preds = model(xb)              # forward pass
        loss = criterion(preds, yb)    # MSE loss for this batch
        loss.backward()                # backward pass (compute gradients)
        optimizer.step()               # gradient-descent update

        epoch_loss += loss.item() * xb.size(0)

    epoch_loss /= len(train_dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:4d}/{EPOCHS}]  Train MSE: {epoch_loss:.4f}")

### 6. Loss curve

I want to actually see the loss go down over training, not just trust the printed numbers.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE Loss")
plt.title("Training Loss over 1000 Epochs")
plt.grid(alpha=0.3)
plt.show()

### 7. Checking performance on my held-out test set

The real test of whether my model learned anything useful (and not just the training data) is how it does on data it's never seen.

In [ ]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        loss = criterion(preds, yb)
        test_loss += loss.item() * xb.size(0)

test_loss /= len(test_dataset)
print(f"Final Test MSE (standardized scale): {test_loss:.4f}")

# let me look at a handful of individual predictions vs. the actual targets,
# converted back to the original scale so the numbers are easier to interpret
model.eval()
with torch.no_grad():
    sample_X = torch.tensor(X_test[:5], dtype=torch.float32)
    sample_pred_scaled = model(sample_X).numpy()
    sample_true_scaled = y_test[:5]

sample_pred = y_scaler.inverse_transform(sample_pred_scaled)
sample_true = y_scaler.inverse_transform(sample_true_scaled)

print("\nPredicted vs. Actual (original scale):")
for p, t in zip(sample_pred.ravel(), sample_true.ravel()):
    print(f"  predicted = {p:8.2f}   actual = {t:8.2f}")

### What I found

- I trained a small **2-fully-connected-layer** network (`Linear → ReLU → Linear`) on my synthetic regression dataset using `nn.MSELoss()`, `torch.optim.SGD`, mini-batches of size **8**, for **1000 epochs**, exactly as specified.
- My loss curve shows the training MSE steadily decreasing and then flattening out, which tells me the model converged rather than still being mid-training or stuck.
- My test-set predictions track the true targets reasonably well, which confirms the model actually generalized to unseen data instead of just memorizing the training set.
